# Bloque 1 — Estructuras de Datos, Expresiones Regulares y Funciones Lambda
---
## Proyecto: Red Social Ficticia

Trabajamos con los datos de una **red social ficticia** generada para practicar análisis de datos con Python. Los datos están repartidos en tres ficheros dentro de la carpeta `data/`:

- **`usuarios.csv`** — un registro por usuario, con campos como 
    - `id_usuario` — identificador único del usuario (numérico como string, p. ej. '1'). Se usa para relacionar con publicaciones.csv y conexiones.json.
    - `nombre` — nombre de pila (p. ej. 'Ana').
    - `apellidos` — uno o dos apellidos (p. ej. 'Fernandez Gonzalez').
    - `edad` — edad del usuario en años, como string (p. ej. '24'') → son str(), conviértela con int()
    - `ciudad` — ciudad de residencia; puede venir vacía ('') si el usuario no la indicó.
    - `pais` — país de residencia (en este dataset, siempre 'Espana').
    - `genero` — código de género, 'F' o 'M'.
    - `fecha_registro` — fecha en que el usuario se dio de alta, formato 'YYYY-MM-DD'.
    - `tipo_cuenta` — tipo de cuenta: 'free', 'premium', 'creator', etc.
    - `seguidores` — número de seguidores, como string (p. ej. '2016') → son str(),conviértelo con int()
    - `seguidos` — número de cuentas a las que sigue, también como string.
    - `publicaciones_totales` — número total de publicaciones hechas por el usuario, como string.
    - `activo` — indica si la cuenta está activa, como string 'True'/'False' (compáralo con == 'True').
    - `bio` — texto libre de biografía/descripción del perfil.


- **`publicaciones.csv`** — un registro por publicación, con 

    - `id_publicacion` — identificador único de la publicación (numérico como string).
    - `id_usuario` — id del usuario que hizo la publicación; es la clave para relacionar cada publicación con su autor en usuarios.csv.
    - `fecha` — fecha de publicación, formato 'YYYY-MM-DD' (se puede ordenar directamente como string).
    - `tipo_contenido` — tipo de contenido publicado: 'foto', 'video', 'texto', etc.
    - `likes` — número de "me gusta", como string con formato decimal (p. ej. '1329.0') → conviértelo con float() (no int() directamente, porque tiene el .0).
    - `comentarios` — número de comentarios, también como string decimal (p. ej. '354.0') → igual, usa float().
    - `compartidos` — número de veces compartida, como string entero (p. ej. '105') → puedes usar int().
    - `visualizaciones` — número de visualizaciones, como string entero.
    - `hashtags` — cadena con los hashtags de la publicación separados por comas (p. ej. '#datascience,#cloud'); para trabajar con ellos individualmente hay que hacer .split(','). Puede venir vacía si la publicación no tiene hashtags.

- **`conexiones.json`** — relaciones de seguimiento entre usuarios: una lista de objetos       
    - `id_usuario` — id del usuario (aquí sí es un número entero real, no un string, porque viene de JSON, no de CSV).
    - `siguiendo` — lista de ids (enteros) de los usuarios a los que este usuario sigue. Es decir, representa las relaciones de "seguir" en la red social, una por cada usuario. `{"id_usuario": ..., "siguiendo": [ids de usuarios a los que sigue]}`.

- **Punto clave:**
    
    - Todos los campos numéricos y booleanos llegan como **texto** al leerlos con el módulo `csv` (p. ej. `'2016'` o `'True'`), así que a menudo habrá que convertirlos (`int(...)`) o compararlos como string antes de operar con ellos.
    - Fíjate que `likes` y `comentarios` vienen con decimales `(.0)` mientras que `compartidos` y `visualizaciones` no — así que al convertir usa `float()` para los dos primeros e `int()` para los otros dos (o `int(float(valor)`) si quieres unificar).
    - En el caso de fichero conexiones:
        - Al venir de JSON (no de CSV), los números ya son int, no strings — no hace falta convertirlos con int().
        - No es un diccionario único, sino una lista de diccionarios (uno por usuario), así que para buscar las conexiones de un usuario concreto hay que recorrer la lista y comparar c['id_usuario'] con el id buscado (como se hace en la función siguiendo_de del ejercicio 3.3).
        - Es la estructura que permite calcular cosas como seguidores en común, recomendaciones, o cuántos usuarios siguen a alguien en concreto.

---
### Objetivos
En este bloque trabajamos con los datos de la red social usando **solo la librería estándar de Python** (sin Pandas).

Contenido:
- **Listas**: operaciones, comprensiones, slicing
- **Diccionarios**: índices, agrupaciones, transformaciones
- **Conjuntos**: operaciones de conjunto sobre datos reales
- **Tuplas y namedtuples**: registros inmutables
- **Funciones lambda** con `filter`, `map`, `sorted` y `reduce`
- **Expresiones regulares**: extracción y búsqueda de hashtags con `re`

---
### Bloque de código inicial
Esta celda carga en memoria los tres ficheros de datos de la red social ficticia (usuarios, publicaciones y conexiones) usando solo la librería estándar de Python, sin necesidad de Pandas. 
Busca automáticamente la carpeta `data/` sea cual sea el directorio desde el que se ejecute el notebook, y avisa con un error claro si no encuentra los ficheros. Al final deja listas las variables `usuarios`, `publicaciones` y `conexiones` que se usarán en todos los ejercicios del bloque

In [ ]:
from pathlib import Path
import csv
import json
import re
from functools import reduce
from collections import namedtuple

# Carga de datos con el modulo csv estandar (SIN pandas)
posibles_carpetas_datos = [
    Path("../data"),
    Path("data"),
    Path("Proy_analisis_Red_Social_Ficticia/data"),
]

DATA_DIR = next(
    (carpeta for carpeta in posibles_carpetas_datos if (carpeta / "usuarios.csv").exists()),
    None,
)

if DATA_DIR is None:
    rutas_probadas = "\n".join(str(carpeta / "usuarios.csv") for carpeta in posibles_carpetas_datos)
    raise FileNotFoundError(
        "No se encontro usuarios.csv. Rutas probadas:\n"
        f"{rutas_probadas}\n\n"
        f"Directorio actual del kernel: {Path.cwd()}"
    )

def cargar_csv(ruta):
    with open(ruta, encoding="utf-8") as f:
        return list(csv.DictReader(f))

usuarios = cargar_csv(DATA_DIR / "usuarios.csv")
publicaciones = cargar_csv(DATA_DIR / "publicaciones.csv")

with open(DATA_DIR / "conexiones.json", encoding="utf-8") as f:
    conexiones = json.load(f)

print(f"Carpeta de datos:       {DATA_DIR}")
print(f"Usuarios cargados:      {len(usuarios)}")
print(f"Publicaciones cargadas: {len(publicaciones)}")
print(f"Conexiones cargadas:    {len(conexiones)}")
print(f"\nEjemplo de usuario:")
for k, v in list(usuarios[0].items())[:6]:
    print(f"  {k}: {v}")


: 

---
## Ejercicios básicos
### Sección 1 — Listas

Las listas son la estructura secuencial mutable más usada en Python.

In [ ]:
# ─── Ejercicio 1.1 — Bucle while con índice manual ────────────────────────
# Practica el bucle 'while' con un índice manual (patrón habitual cuando se
# leen datos con input() hasta que el usuario decide terminar).
# Recorre 'usuarios' con un bucle while y un índice 'i' que empiece en 0.
# Añade el nombre de cada usuario a la lista 'nombres_while' con .append(),
# incrementando 'i' en cada vuelta, hasta procesar los 10 primeros usuarios.

nombres_while = []
i = 0
while i < 10:
    nombres_while.append(usuarios[i]['nombre'])
    i += 1

print(nombres_while)

In [ ]:
# ─── Ejercicio 1.2 — Posiciones con range(len) y enumerate ────────────────
# Practica el uso de índices explícitos con range(len(...)).
# Crea una lista de tuplas (posicion, nombre) para los 5 primeros usuarios,
# donde 'posicion' empieza en 0.
# Una vez terminado haz el mismo ejerciio utilizando enumerate() en vez de range(len(...)).

primeros = usuarios[:5]

# Con range(len(...))
posiciones_nombres = []
for i in range(len(primeros)):
    posiciones_nombres.append((i, primeros[i]['nombre']))

# Con enumerate()
posiciones_nombres_enum = [(i, u['nombre']) for i, u in enumerate(primeros)]

print(posiciones_nombres)
print(posiciones_nombres == posiciones_nombres_enum)

In [ ]:
# ─── Ejercicio 1.3 — Acumulador manual: menos seguidores ──────────────────
# Practica el patrón de acumulador manual con variable "flag": calcula el
# usuario con MENOS seguidores SIN usar la función min().
# Recorre 'usuarios' con un for y usa una variable booleana 'primero' para
# inicializar el acumulador 'usuario_menos_seguidores' la primera vuelta.

usuario_menos_seguidores = None
primero = True

for u in usuarios:
    if primero or int(u['seguidores']) < int(usuario_menos_seguidores['seguidores']):
        usuario_menos_seguidores = u
        primero = False

print(usuario_menos_seguidores['nombre'], usuario_menos_seguidores['seguidores'])

In [ ]:
# ─── Ejercicio 1.4 — Slicing de listas ────────────────────────────────────
# Usa SLICING para obtener:
# a) los 5 primeros usuarios
# b) los 10 últimos usuarios
# c) uno de cada tres (cada tercer usuario)

primeros_5  = usuarios[:5]
ultimos_10  = usuarios[-10:]
cada_tres   = usuarios[::3]

print(len(primeros_5), len(ultimos_10), len(cada_tres))

In [ ]:
# ─── Ejercicio 1.5 — Filtrar usuarios activos ─────────────────────────────
# El campo 'activo' viene como string 'True' / 'False' desde el CSV.
# Crea una lista 'activos' con solo los usuarios cuyo campo 'activo' == 'True'.

activos = [u for u in usuarios if u['activo'] == 'True']

print(len(activos))

In [ ]:
# ─── Ejercicio 1.6 — Filtrar usuarios populares ───────────────────────────
# Crea una lista 'populares' con los usuarios que tienen MÁS de 1000 seguidores.
# Nota: el campo 'seguidores' viene como string desde el CSV → convierte a int.

populares = [u for u in usuarios if int(u['seguidores']) > 1000]

print(len(populares))

In [ ]:
# ─── Ejercicio 1.7 — Pares (nombre, seguidores) con comprensión ───────────
# Crea una lista de tuplas (nombre, seguidores) para todos los usuarios.
# Usa comprensión de lista.

pares_nombre_seg = [(u['nombre'], int(u['seguidores'])) for u in usuarios]

print(pares_nombre_seg[:3])

In [ ]:
# ─── Ejercicio 1.8 — Lista plana de hashtags ──────────────────────────────
# El campo 'hashtags' de cada publicación es una cadena '#ia,#python,...'
# Crea una lista PLANA con todos los hashtags de todas las publicaciones.
# Incluye los duplicados (se eliminarán en la sección de conjuntos).

todos_hashtags = [
    h
    for p in publicaciones
    for h in p['hashtags'].split(',')
    if h
]

print(len(todos_hashtags), todos_hashtags[:5])

---
## Sección 2 — Diccionarios

Los diccionarios permiten acceso O(1) por clave y son perfectos para crear índices.

In [ ]:
# ─── Ejercicio 2.1 — Índice de usuarios por id ────────────────────────────
# Crea un diccionario 'indice_usuarios' que mapee id_usuario → nombre.
# Así podemos buscar el nombre de un usuario por su id en O(1).

indice_usuarios = {u['id_usuario']: u['nombre'] for u in usuarios}

print(indice_usuarios['1'])

In [ ]:
# ─── Ejercicio 2.2 — Agrupar nombres por ciudad ───────────────────────────
# Crea un diccionario 'por_ciudad' que mapee ciudad → [lista de nombres].
# Ignora los usuarios sin ciudad (campo vacío o None).

por_ciudad = {}
for u in usuarios:
    ciudad = u['ciudad']
    if not ciudad:
        continue
    por_ciudad.setdefault(ciudad, []).append(u['nombre'])

print(len(por_ciudad), 'ciudades')

In [ ]:
# ─── Ejercicio 2.3 — Buscar un usuario por su id ──────────────────────────
# Crea una función que reciba un id_usuario (como string) y devuelva
# el diccionario completo del usuario usando el índice creado en 2.1.

# indice_usuarios (2.1) solo guarda el nombre. Para devolver el diccionario
# completo construimos un segundo índice id_usuario → usuario.
indice_completo = {u['id_usuario']: u for u in usuarios}

def buscar_usuario(id_str):
    """Devuelve el dict del usuario con ese id, o None si no existe."""
    return indice_completo.get(id_str)

print(buscar_usuario('1')['nombre'], buscar_usuario('9999'))

In [ ]:
# ─── Ejercicio 2.4 — Campo calculado sobre una copia ──────────────────────
# Añade un campo calculado 'ratio_engagement' a cada usuario.
# ratio_engagement = seguidores / (seguidos + 1)  (el +1 evita división por cero)
# Trabaja sobre una COPIA de la lista para no modificar el original.

usuarios_con_ratio = []
for u in usuarios:
    u_copia = dict(u)  # copia del diccionario
    u_copia['ratio_engagement'] = int(u['seguidores']) / (int(u['seguidos']) + 1)
    usuarios_con_ratio.append(u_copia)

print(round(usuarios_con_ratio[0]['ratio_engagement'], 2), 'ratio_engagement' in usuarios[0])

---
## Sección 3 — Conjuntos

Los conjuntos (`set`) garantizan unicidad y soportan operaciones de teoría de conjuntos.

In [ ]:
# ─── Ejercicio 3.1 — Conjunto de ciudades únicas ──────────────────────────
# Crea un set 'ciudades_unicas' con todas las ciudades distintas de los usuarios.
# Ignora los valores vacíos.

ciudades_unicas = {u['ciudad'] for u in usuarios if u['ciudad']}

print(len(ciudades_unicas), sorted(ciudades_unicas))

In [ ]:
# ─── Ejercicio 3.2 — Hashtags únicos frente a totales ─────────────────────
# Crea un set 'hashtags_unicos' a partir de la lista 'todos_hashtags' del ejercicio 1.8.
# Muestra cuántos hashtags únicos hay vs el total con duplicados.

hashtags_unicos_set = set(todos_hashtags)

print(f"Hashtags únicos: {len(hashtags_unicos_set)} de {len(todos_hashtags)} en total")

In [ ]:
# ─── Ejercicio 3.3 — Seguidos en común entre dos usuarios ─────────────────
# Usa el fichero de conexiones para encontrar los usuarios seguidos en COMÚN
# por el usuario con id=1 y el usuario con id=2.
# conexiones es una lista de {id_usuario: int, siguiendo: [int, ...]}

def siguiendo_de(id_usuario):
    """Devuelve el set de ids que sigue el usuario dado."""
    for c in conexiones:
        if c['id_usuario'] == id_usuario:
            return set(c['siguiendo'])
    return set()

sig_1 = siguiendo_de(1)
sig_2 = siguiendo_de(2)

comunes = sig_1 & sig_2  # <-- sig_1 & sig_2  (intersección)
solo_1  = sig_1 - sig_2  # <-- sig_1 - sig_2  (diferencia)

print('En común:', sorted(comunes))
print('Solo el usuario 1:', sorted(solo_1))

In [ ]:
# ─── Ejercicio 3.4 — Usuarios seguidos por al menos 3 personas ────────────
# Encuentra los usuarios que son seguidos por AL MENOS 3 usuarios distintos.
# Itera sobre todas las conexiones y cuenta cuántas veces aparece cada id.

todos_seguidos = []
for c in conexiones:
    todos_seguidos.extend(c['siguiendo'])

# Cuántos usuarios distintos siguen a cada id (cada id aparece una vez por seguidor)
conteo_seguidores = {}
for id_seguido in todos_seguidos:
    conteo_seguidores[id_seguido] = conteo_seguidores.get(id_seguido, 0) + 1

populares_seguidos = {id_u for id_u, n in conteo_seguidores.items() if n >= 3}

print(len(populares_seguidos), 'usuarios seguidos por al menos 3 personas')

---
## Sección 4 — Tuplas y namedtuples

Las tuplas son inmutables. Las `namedtuple` añaden nombres a los campos, haciendo el código más legible.

In [ ]:
# ─── Ejercicio 4.1 — namedtuple de usuario ────────────────────────────────
# Define un namedtuple 'RegistroUsuario' con los campos:
#   id_usuario, nombre, ciudad, seguidores, tipo_cuenta
# Crea una lista de RegistroUsuario a partir de 'usuarios'.

RegistroUsuario = namedtuple(
    'RegistroUsuario',
    ['id_usuario', 'nombre', 'ciudad', 'seguidores', 'tipo_cuenta'],
)

registros = [
    RegistroUsuario(
        u['id_usuario'],
        u['nombre'],
        u['ciudad'],
        int(u['seguidores']),
        u['tipo_cuenta'],
    )
    for u in usuarios
]

print(registros[0])

In [ ]:
# ─── Ejercicio 4.2 — Inmutabilidad de las tuplas ──────────────────────────
# Demuestra la INMUTABILIDAD de las tuplas:
# Intenta modificar el campo 'seguidores' del primer registro.
# Captura la excepción con try/except y muestra el mensaje de error.

try:
    registros[0].seguidores = 0
except AttributeError as e:
    print('Error al modificar la tupla:', e)

In [ ]:
# ─── Ejercicio 4.3 — Desempaquetado de tuplas ─────────────────────────────
# Usa DESEMPAQUETADO de tupla para extraer los campos del primer registro.
# Asigna cada campo a una variable distinta.

id_u, nombre_u, ciudad_u, seg_u, tipo_u = registros[0]

print(id_u, nombre_u, ciudad_u, seg_u, tipo_u)

---
## Sección 5 — Funciones Lambda

`lambda`, `filter`, `map`, `sorted` y `reduce` permiten procesar listas de forma concisa y funcional.

In [ ]:
# ─── Ejercicio 5.1 — Ordenar usuarios con sorted y lambda ─────────────────
# Ordena 'usuarios' por número de seguidores de MAYOR a MENOR usando sorted() + lambda.

usuarios_por_seg = sorted(usuarios, key=lambda u: int(u['seguidores']), reverse=True)

print(usuarios_por_seg[0]['nombre'], usuarios_por_seg[0]['seguidores'])

In [ ]:
# ─── Ejercicio 5.2 — Filtrar creators con filter y lambda ─────────────────
# Usa filter() + lambda para obtener solo los usuarios de tipo 'creator'.

creators = list(filter(lambda u: u['tipo_cuenta'] == 'creator', usuarios))

print(len(creators))

In [ ]:
# ─── Ejercicio 5.3 — Etiquetar usuarios con map ───────────────────────────
# Usa map() + lambda para crear una lista de etiquetas:
#   Si seguidores >= 5000 → 'influencer', si >= 1000 → 'popular', si no → 'normal'

def etiqueta(u):
    seguidores = int(u['seguidores'])
    if seguidores >= 5000:
        return 'influencer'
    if seguidores >= 1000:
        return 'popular'
    return 'normal'

# map recibe la función directamente (equivale a lambda u: etiqueta(u))
etiquetas = list(map(etiqueta, usuarios))

print(etiquetas[:5])

In [ ]:
# ─── Ejercicio 5.4 — Total de seguidores con reduce ───────────────────────
# Usa reduce() para calcular el TOTAL de seguidores de todos los usuarios.

total_seguidores = reduce(lambda acumulado, u: acumulado + int(u['seguidores']), usuarios, 0)

print(total_seguidores)

In [ ]:
# ─── Ejercicio 5.5 — Ordenar publicaciones por fecha ──────────────────────
# Ordena las publicaciones por fecha de publicación (de más antigua a más reciente).
# El campo 'fecha' es una cadena 'YYYY-MM-DD' → se puede ordenar como string.

pubs_ordenadas = sorted(publicaciones, key=lambda p: p['fecha'])

print(pubs_ordenadas[0]['fecha'], pubs_ordenadas[-1]['fecha'])

---
## Sección 6 — Expresiones regulares

El módulo estándar `re` permite localizar patrones de texto. En una red social resulta útil para extraer etiquetas y buscar publicaciones concretas.

Patrones que usaremos:

```python
r'#[\w]+'             # hashtag: # seguido de letras, números o _
re.findall(patron, texto)
re.search(patron, texto, flags=re.IGNORECASE)
```

Usa cadenas *raw* (`r'...'`) para que las barras invertidas del patrón se interpreten correctamente.

In [ ]:
# ─── Ejercicio R.1 — Extraer hashtags con findall ─────────────────────────
# Extrae todos los hashtags de este texto usando re.findall().
# El patrón debe encontrar # seguido de letras, números o _. Normaliza el
# resultado a minúsculas y conserva el orden, incluidos los duplicados.

texto = 'Aprende #Python, #IA y #datos_2026. También #python.'
hashtags_regex = [h.lower() for h in re.findall(r'#\w+', texto)]

print(hashtags_regex)

In [ ]:
# ─── Ejercicio R.2 — Buscar un hashtag completo ───────────────────────────
# Filtra las publicaciones que contienen exactamente el hashtag buscado.
# La búsqueda debe ignorar mayúsculas, pero '#ia' NO debe encontrar '#iabd'.
# Pista: re.escape(hashtag) convierte el hashtag en un patrón seguro y
# (?!\w) evita que después continúen letras, números o _.

hashtag_buscado = '#ia'
patron = re.compile(re.escape(hashtag_buscado) + r'(?!\w)', flags=re.IGNORECASE)
pubs_con_ia = [p for p in publicaciones if patron.search(p['hashtags'])]

print(len(pubs_con_ia), 'publicaciones con', hashtag_buscado)

---
## Entrega — `../scripts/pipeline.py` (Bloque 1)

Implementa las siguientes funciones en `../scripts/pipeline.py` y ejecuta:
```bash
python -m pytest ../tests/test_bloque1.py -v
```

| Función | Descripción |
|---|---|
| `usuarios_activos(lista)` | Filtra usuarios con `activo == 'True'` |
| `agrupar_por_ciudad(lista)` | Devuelve `{ciudad: [nombres]}` |
| `hashtags_unicos(lista_pub)` | Devuelve `set` de hashtags únicos |
| `top_por_seguidores(lista, n)` | Devuelve los n usuarios con más seguidores |
| `extraer_hashtags_regex(texto)` | Extrae y normaliza hashtags con `re.findall` |
| `publicaciones_con_hashtag(lista_pub, hashtag)` | Filtra por hashtag completo sin distinguir mayúsculas |

## Ejercicios avanzados

Estos ejercicios combinan varias estructuras y funciones. Resuélvelos después de completar los ejercicios básicos.

### A.1 Informe de actividad por ciudad
Crea un diccionario anidado con esta estructura:

```python
{
    'Madrid': {
        'usuarios': 12,
        'activos': 10,
        'seguidores_totales': 25000,
        'media_seguidores': 2083.33
    }
}
```

Agrupa los usuarios por ciudad, ignora ciudades vacías y calcula los cuatro valores usando listas, diccionarios y `lambda`.

In [ ]:
def informe_actividad_ciudades(lista_usuarios):
    """Resume usuarios, activos y seguidores por ciudad (ignora ciudades vacías)."""
    por_ciudad_usuarios = {}
    for u in lista_usuarios:
        if u['ciudad']:
            por_ciudad_usuarios.setdefault(u['ciudad'], []).append(u)

    informe = {}
    for ciudad, grupo in por_ciudad_usuarios.items():
        seguidores_totales = sum(map(lambda u: int(u['seguidores']), grupo))
        informe[ciudad] = {
            'usuarios': len(grupo),
            'activos': len(list(filter(lambda u: u['activo'] == 'True', grupo))),
            'seguidores_totales': seguidores_totales,
            'media_seguidores': round(seguidores_totales / len(grupo), 2),
        }
    return informe

informe_ciudades = informe_actividad_ciudades(usuarios)
for ciudad in sorted(informe_ciudades)[:3]:
    print(ciudad, informe_ciudades[ciudad])

### A.2 Recomendaciones por conexiones
Para cada usuario, encuentra hasta cinco recomendaciones: personas que sigue alguno de sus contactos pero que todavía no sigue él. No incluyas al propio usuario y ordena las recomendaciones por número de seguidores.


In [ ]:
def recomendaciones_por_conexiones(lista_usuarios, conexiones, limite=5):
    """Devuelve {id_usuario: [ids recomendados]} con hasta 'limite' ids por usuario."""
    siguiendo = {c['id_usuario']: set(c['siguiendo']) for c in conexiones}
    seguidores = {int(u['id_usuario']): int(u['seguidores']) for u in lista_usuarios}

    recomendaciones = {}
    for id_usuario, seguidos in siguiendo.items():
        # Personas que siguen mis contactos...
        candidatos = set().union(*(siguiendo.get(contacto, set()) for contacto in seguidos))
        # ...menos las que ya sigo y yo mismo
        candidatos -= seguidos
        candidatos.discard(id_usuario)
        # Mayor número de seguidores primero (desempate por id)
        ordenados = sorted(
            (c for c in candidatos if c in seguidores),
            key=lambda c: (-seguidores[c], c),
        )
        recomendaciones[id_usuario] = ordenados[:limite]
    return recomendaciones

recomendaciones = recomendaciones_por_conexiones(usuarios, conexiones)
print(recomendaciones[1])

### A.3 Pipeline funcional de publicaciones
Construye una función que reciba la lista de publicaciones y devuelva las cinco etiquetas más utilizadas. Debe encadenar `map`, `filter`, `sorted` y un diccionario de conteos. Compara el resultado con una solución basada en conjuntos.


In [ ]:
def top_hashtags(lista_publicaciones, n=5):
    """Las n etiquetas más usadas como lista de tuplas (hashtag, veces)."""
    listas = map(lambda p: p['hashtags'].split(','), lista_publicaciones)
    planos = (h for lista in listas for h in lista)
    validos = filter(lambda h: h.strip() != '', planos)

    conteo = {}
    for h in validos:
        conteo[h] = conteo.get(h, 0) + 1

    # Más usadas primero; desempate alfabético para que el resultado sea estable
    return sorted(conteo.items(), key=lambda par: (-par[1], par[0]))[:n]

top_funcional = top_hashtags(publicaciones)
print(top_funcional)

# Solución basada en conjuntos: el set da los hashtags únicos y para cada uno
# se cuentan las publicaciones que lo contienen.
unicos = {h for p in publicaciones for h in p['hashtags'].split(',') if h}
conteo_conjuntos = {
    h: sum(1 for p in publicaciones if h in p['hashtags'].split(','))
    for h in unicos
}
top_conjuntos = sorted(conteo_conjuntos.items(), key=lambda par: (-par[1], par[0]))[:5]

print(top_funcional == top_conjuntos)

### A.4 Limpieza de texto
Crea una función `normalizar_hashtag(texto)` que elimine espacios, convierta a minúsculas y garantice que el resultado comienza por `#`. Aplica la función con `map` y elimina los resultados vacíos o repetidos.

In [ ]:
def normalizar_hashtag(texto):
    """Sin espacios, en minúsculas y con un único '#' al principio ('' si no queda nada)."""
    limpio = ''.join(texto.split()).lower()
    limpio = limpio.lstrip('#')
    return '#' + limpio if limpio else ''

# Lista con hashtags mal escritos (esto es solo una prueba)
hashtags_sucios = [' #IA ', 'Python', '# Datos', '#ia', '', '   ', '#PYTHON', 'datos']
normalizados = map(normalizar_hashtag, hashtags_sucios)
sin_vacios = filter(lambda h: h != '', normalizados)
hashtags_limpios = list(dict.fromkeys(sin_vacios))  # quita repetidos conservando el orden
print(hashtags_limpios)

# Aplicado a los hashtags reales de las publicaciones
hashtags_reales = list(dict.fromkeys(
    filter(lambda h: h != '', map(normalizar_hashtag, todos_hashtags))
))
print(len(hashtags_reales), len(hashtags_unicos_set))

### A.5 Validación de publicaciones
Comprueba que todos los ids de las publicaciones pertenecen a usuarios existentes y devuelve dos listas: publicaciones válidas y publicaciones con errores. No modifiques las listas originales.

In [ ]:
def validar_publicaciones(lista_publicaciones, lista_usuarios):
    """Devuelve (validas, con_error) sin modificar las listas originales."""
    ids_usuarios = {u['id_usuario'] for u in lista_usuarios}
    validas = [p for p in lista_publicaciones if p['id_usuario'] in ids_usuarios]
    con_error = [p for p in lista_publicaciones if p['id_usuario'] not in ids_usuarios]
    return validas, con_error

validas, con_error = validar_publicaciones(publicaciones, usuarios)
print(len(validas), len(con_error))

# Prueba con una publicación de un usuario que no existe (lista nueva, no se toca la original)
pubs_prueba = publicaciones + [{'id_publicacion': '9999', 'id_usuario': '9999'}]
validas_p, con_error_p = validar_publicaciones(pubs_prueba, usuarios)
print(len(validas_p), len(con_error_p), len(publicaciones))